In [0]:
from pyspark.sql.functions import *
from datetime import datetime,date


In [0]:
dbutils.widgets.text("load_date", "")
load_date = dbutils.widgets.get("load_date")

if load_date:
    load_date=date.fromisoformat(load_date)
else:
    load_date=date.today()
# base_path = "/Volumes/workspace/default/fhir_files/raw/patient/"
# target_path = f"{base_path}extraction_date={load_date}/"

In [0]:
%run ./config_notebook

In [0]:
# Reads raw FHIR JSON files for each resource type and loads them into Bronze Delta tables with extraction metadata.
for resource_name,config in raw_filepaths.items():
    target_path=f"{config['raw_path']}/extraction_date={load_date}"
    
    df=spark.read.format("json").option("multiLine","true").load(target_path)
    
    if resource_name=="Patient":
        df_patient=df
        df_patient = df.select(explode("entry").alias("entry"),"_metadata.file_path")
        df_patient = df_patient.select("entry.resource.*","file_path").withColumn("extraction_timestamp",to_timestamp(lit(load_date))).withColumn("api_url_or_params",lit("Patient?_lastUpdated=last_3_days"))
        df_patient.write.format('delta').mode('overwrite').saveAsTable(config['table_name'])

    elif resource_name=="Encounter":
        df_encounter=df
        df_encounter = df.select(explode("entry").alias("entry"),"_metadata.file_path")
        df_encounter = df_encounter.select("entry.resource.*","file_path").withColumn("extraction_timestamp",to_timestamp(lit(load_date))).withColumn("api_url_or_params",lit("encounter?_lastUpdated=last_3_days")) 
        df_encounter.write.format('delta').mode('overwrite').saveAsTable(config['table_name'])      
    elif resource_name=="Observation":
        df_observation=df
        df_observation = df.select(explode("entry").alias("entry"),"_metadata.file_path")
        df_observation = df_observation.select("entry.resource.*","file_path").withColumn("extraction_timestamp",to_timestamp(lit(load_date))).withColumn("api_url_or_params",lit("observation?_lastUpdated=last_3_days"))
        df_observation.write.format('delta').mode('overwrite').saveAsTable(config['table_name'])
    elif resource_name=="Condition":
        df_condition=df
        df_condition = df.select(explode("entry").alias("entry"),"_metadata.file_path")
        df_condition = df_condition.select("entry.resource.*","file_path").withColumn("extraction_timestamp",to_timestamp(lit(load_date))).withColumn("api_url_or_params",lit("condition?_lastUpdated=last_3_days"))
        df_condition.write.format('delta').mode('overwrite').saveAsTable(config['table_name'])
    
    print(resource_name,"count:",df.count())

In [0]:
dbutils.notebook.exit("Success")